<a href="https://www.kaggle.com/code/avikdas567/vesuvius-challenge-surface-detection?scriptVersionId=293593791" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# Vesuvius Challenge - Surface Detection
# ============================================================

import os
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import tifffile as tiff

# ----------------------------
# Paths
# ----------------------------
INPUT_DIR = "/kaggle/input/vesuvius-challenge-surface-detection"
TEST_IMG_DIR = os.path.join(INPUT_DIR, "test_images")
OUTPUT_DIR = "/kaggle/working/preds"
ZIP_PATH = "/kaggle/working/submission.zip"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------
# Load test metadata
# ----------------------------
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
print(f"Found {len(test_df)} test volumes")

# ----------------------------
# LZW-safe TIFF reader
# ----------------------------
def read_tiff_stack_pil(path):
    imgs = []
    with Image.open(path) as img:
        i = 0
        while True:
            try:
                img.seek(i)
                imgs.append(np.array(img))
                i += 1
            except EOFError:
                break
    return np.stack(imgs, axis=0)  # (Z, Y, X)

# ----------------------------
# Surface detection
# ----------------------------
def improved_surface_detection(volume):
    """
    Gradient-based surface detection with smoothing + thickness
    """
    Z, Y, X = volume.shape
    mask = np.zeros((Z, Y, X), dtype=np.uint8)

    # normalize
    v = volume.astype(np.float32)
    v = (v - v.min()) / (v.max() - v.min() + 1e-6)

    # compute gradient along Z (surface = strong transition)
    grad = np.abs(np.diff(v, axis=0, prepend=v[:1]))

    # smooth gradient slightly along Z
    for z in range(1, Z - 1):
        grad[z] = (grad[z - 1] + grad[z] + grad[z + 1]) / 3.0

    # adaptive threshold
    global_thresh = np.quantile(grad, 0.92)

    for y in range(Y):
        for x in range(X):
            col = grad[:, y, x]
            idx = np.argmax(col)
            if col[idx] > global_thresh:
                # thickness along Z
                for dz in (-2, -1, 0, 1, 2):
                    z = idx + dz
                    if 0 <= z < Z:
                        mask[z, y, x] = 1

    return mask

# ----------------------------
# Run inference
# ----------------------------
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    vid = row["id"]
    img_path = os.path.join(TEST_IMG_DIR, f"{vid}.tif")

    volume = read_tiff_stack_pil(img_path)
    pred = improved_surface_detection(volume)

    tiff.imwrite(
        os.path.join(OUTPUT_DIR, f"{vid}.tif"),
        pred.astype(np.uint8)
    )

# ----------------------------
# Zip submission
# ----------------------------
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, fname), fname)

# ----------------------------
# Final check
# ----------------------------
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Submission created!")
    print("Files:", len(z.namelist()))
    print("Example:", z.namelist()[:5])

print("READY TO SUBMIT:", ZIP_PATH)

Found 1 test volumes


100%|██████████| 1/1 [00:03<00:00,  3.38s/it]


Submission created!
Files: 1
Example: ['1407735.tif']
READY TO SUBMIT: /kaggle/working/submission.zip
